In [1]:
import argparse
import os,cv2,math
import random,numpy,pandas
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

In [2]:
import argparse
import os
import random,numpy
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.optim as optim
import torch.nn.functional as F
import torch.utils.data
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torchvision
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torchvision.utils as vutils
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

In [3]:
seed = 999
print("Random Seed: ", seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

Random Seed:  999


In [4]:
workers = 2
batch_size = 10
nz = 100
num_epochs = 5
lr = 0.0001
beta1 = 0.5
ngpu=1
ngf,nc = 3,3
ndf = 64

transform = transforms.Compose([
    transforms.Resize(512),
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

trainset = torchvision.datasets.ImageFolder(root=f'/kaggle/input/fake-scene-dataset/train',transform=transform)
train,test = torch.utils.data.random_split(trainset, [len(trainset)-100,100]) 

dataloader = torch.utils.data.DataLoader(train,batch_size=batch_size,shuffle=True,num_workers=2)
testloader = torch.utils.data.DataLoader(test,batch_size=batch_size,shuffle=True,num_workers=2)

#print(len(dataloader),len(testloader))

device = torch.device("cuda:0" if (torch.cuda.is_available() and ngpu > 0) else "cpu")

In [5]:
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

class FSC(nn.Module):
    def __init__(self):
        super().__init__()
        self.FSC_rafire=nn.Sequential(
            nn.Conv2d(3, 16, 3),  # Increased channels
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(16, 64, 3),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(64, 126, 3),
            nn.BatchNorm2d(126),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(126, 256, 3),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(256, 512, 3),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(512, 1024, 3),
            nn.BatchNorm2d(1024),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(1024, 2048, 3),
            nn.BatchNorm2d(2048),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Flatten(),
            
            nn.Linear(8192, 2),
            nn.Sigmoid()
        )

    def forward(self,x):
        rsc_rafire = self.FSC_rafire(x)
        return  rsc_rafire,torch.argmax(rsc_rafire, dim=1)

In [6]:
EFF_NET = FSC().to(device)
if (device.type == 'cuda') and (ngpu > 1):
    EFF_NET = nn.DataParallel(EFF_NET, list(range(ngpu)))
EFF_NET.apply(weights_init)

FSC(
  (FSC_rafire): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1))
    (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(16, 64, kernel_size=(3, 3), stride=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 126, kernel_size=(3, 3), stride=(1, 1))
    (9): BatchNorm2d(126, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (12): Conv2d(126, 256, kernel_size=(3, 3), stride=(1, 1))
    (13): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (14): ReLU()
    (15): MaxPool2d(kernel_size=2, stride=2, paddi

In [7]:
criterion = nn.CrossEntropyLoss()

# CrossEntropyLoss
# MSELoss
# L1Loss
# BCELoss
# BCEWithLogitsLoss

optimizer = optim.AdamW(EFF_NET.parameters(), lr=lr, betas=(beta1, 0.999))
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.86)

In [8]:
print(train.dataset.classes)
z_w_=[]
high_rf,i_w,z_k=0,0,0
high_rf_=1

lab={
    'editada' : 0,
    'real' : 1
}
fsc_submission=pandas.read_csv("/kaggle/input/cidaut-ai-fake-scene-classification-2024/sample_submission.csv", index_col ="image")#.to_dict('list')

while(i_w<4000):
    z_,z,z_w=0,0,{}
    correct=0
    for i, data in enumerate(dataloader, 0):
        real=data[0].to(device)
        label=data[1].to(device) #
        #print(data[0].shape)
        raw_label=torch.from_numpy(numpy.array([[0,1] if i==1 else [1,0] for i in data[1] ])).float().to(device)
        raw_output,output = EFF_NET(real)
        raw_output,output = raw_output.float() ,torch.tensor([lab[train.dataset.classes[i]] for i in output.view(-1)])
        #print(raw_label,raw_output)
        err_real = criterion(raw_output,raw_label) #err_real.requires_grad = True
        optimizer.zero_grad()
        err_real.backward()
        optimizer.step()
        
    for i, data in enumerate(testloader, 0):
        real=data[0].to(device)
        label=data[1].cpu()
        raw_output,output = EFF_NET(real)
        output=torch.tensor([lab[train.dataset.classes[i]] for i in output.view(-1)]).cpu()
        z_+=len(output)
        correct+=(output==label).float().sum()

    print(output,label)
        
    acc=100*correct/z_
    z_w_.append(acc)
    
    print(f"EPOCH: {z_k} LOSS_FSC: {err_real.item()} ACCURACY: {acc}")
    #print(z_)	
        
    if len(z_w_)>=3:
        if len([True for i in range(1,4) if z_w_[len(z_w_)-i]<=z_w_[len(z_w_)-3] and z_w_[len(z_w_)-i]>=z_w_[len(z_w_)-4]])==3:
            z_w_=[]
            print(optimizer.param_groups[0]["lr"])
            scheduler.step()
            print(optimizer.param_groups[0]["lr"])
    
    torch.save(EFF_NET.state_dict(),f'/kaggle/working/{err_real.item()} {acc}.pth')# {z_add}
    
    if z_k==100:
        break
    z_k+=1
    i_w+=1
    

['editada', 'real']
tensor([0, 1, 1, 1, 0, 1, 0, 0, 1, 1]) tensor([1, 1, 0, 1, 1, 1, 1, 0, 1, 1])
EPOCH: 0 LOSS_FSC: 0.6291585564613342 ACCURACY: 57.0
tensor([0, 1, 0, 1, 1, 1, 1, 0, 1, 1]) tensor([0, 1, 0, 0, 1, 1, 1, 1, 1, 0])
EPOCH: 1 LOSS_FSC: 0.6292538046836853 ACCURACY: 59.0
tensor([0, 1, 0, 0, 1, 1, 1, 0, 0, 0]) tensor([0, 1, 0, 0, 0, 1, 1, 0, 0, 1])
EPOCH: 2 LOSS_FSC: 0.724059522151947 ACCURACY: 61.0
tensor([0, 0, 1, 1, 1, 1, 0, 0, 0, 1]) tensor([1, 0, 1, 1, 0, 1, 1, 1, 0, 0])
EPOCH: 3 LOSS_FSC: 0.5153998732566833 ACCURACY: 65.0
tensor([1, 1, 0, 1, 1, 1, 0, 1, 1, 0]) tensor([1, 1, 0, 0, 1, 1, 1, 1, 1, 1])
EPOCH: 4 LOSS_FSC: 0.6719784140586853 ACCURACY: 73.0
tensor([0, 0, 1, 0, 1, 1, 0, 1, 0, 1]) tensor([1, 0, 1, 1, 1, 0, 0, 1, 1, 1])
EPOCH: 5 LOSS_FSC: 0.5077357292175293 ACCURACY: 73.0
tensor([0, 0, 1, 1, 0, 1, 0, 1, 1, 1]) tensor([0, 1, 1, 0, 1, 0, 1, 1, 1, 1])
EPOCH: 6 LOSS_FSC: 0.41488590836524963 ACCURACY: 69.0
0.0001
8.6e-05
tensor([0, 1, 0, 0, 1, 0, 1, 1, 1, 1]) tensor([0